<div style="max-width:900px; margin:40px auto 30px auto; padding:30px 24px;
            text-align:center; font-family:Arial, Helvetica, sans-serif;">

  <div style="font-size:15px; font-weight:600; letter-spacing:0.18em;
              text-transform:uppercase; color:#555; margin-bottom:8px;">
    EPIC Jr II 2026
  </div>

  <div style="font-size:18px; letter-spacing:0.08em;
              color:#777; margin-bottom:26px;">
    Hands-On Astro
  </div>

  <div style="width:70px; height:2px; background:#333;
              margin:0 auto 26px auto;"></div>

  <div style="font-size:42px; font-weight:700; line-height:1.15;
              color:#222; margin-bottom:50px;">
    El chirrido de siete milisegundos
  </div>

  <div style="font-size:13px; letter-spacing:0.10em;
              text-transform:uppercase; color:#888; margin-bottom:6px;">
    Autores del Proyecto
  </div>

  <div style="font-size:16px; color:#444; line-height:1.7;">
    Franklin Limacho (Yachay Tech) &nbsp;&middot;&nbsp; Sebastián Castañeda (Yachay Tech) &nbsp;&middot;&nbsp; Jorge Pico (Yachay Tech)
  </div>

</div>

<div style="max-width:850px; margin:20px auto 45px auto; padding:28px 36px;
            text-align:center; font-family:Arial, Helvetica, sans-serif;
            border-top:1px solid #d8d8d8;
            border-bottom:1px solid #d8d8d8;">

  <div style="font-size:13px; font-weight:600; letter-spacing:0.16em;
              text-transform:uppercase; color:#777; margin-bottom:14px;">
    Pregunta científica principal
  </div>

  <div style="font-size:26px; font-weight:500; line-height:1.45;
              color:#222; margin-bottom:30px;">
    ¿Puede reconocerse la misma señal de ondas gravitacionales en dos detectores y qué nos dice la diferencia entre su tiempo de llegada?
  </div>

  <div style="width:50px; height:1px; background:#c8c8c8;
              margin:0 auto 24px auto;"></div>

  <div style="font-size:12px; font-weight:600; letter-spacing:0.14em;
              text-transform:uppercase; color:#888; margin-bottom:13px;">
    Equipo investigador
  </div>

  <div style="font-size:19px; font-weight:600; line-height:1.7;
              color:#333;">
    Nombre 1 &nbsp;·&nbsp;&nbsp;
    Nombre 2 &nbsp;&nbsp;·&nbsp;&nbsp;
    Nombre 3
  </div>

</div>

## Cuaderno del estudiante

**GW150914** fue la primera detección directa de ondas gravitacionales, registrada el 14 de septiembre de 2015 por los detectores LIGO de Hanford (H1) y Livingston (L1). En este cuaderno van a recuperar, con sus propios datos y su propio código, una pieza central de esa evidencia: ¿aparece una señal relacionada en los dos detectores, y qué nos dice la diferencia en el tiempo de llegada?

Los datos que van a usar ya fueron descargados y acondicionados por el equipo del proyecto a partir de los archivos oficiales de GWOSC (ver `student_guide.md` para más contexto). Ustedes no necesitan descargar nada, leer archivos HDF5, ni diseñar filtros.

<div style="width:72%; margin:55px 0 30px 0; padding:3px 0 14px 18px;
            font-family:Arial, Helvetica, sans-serif;
            border-left:3px solid #526b84;">

  <div style="font-size:15px; font-weight:700; letter-spacing:0.14em;
              text-transform:uppercase; color:#526b84; margin-bottom:7px;">
    Sesión 1
  </div>

  <div style="font-size:30px; font-weight:650; line-height:1.25;
              color:#222; margin-bottom:6px;">
    LIGO y las ondas gravitacionales
  </div>

  <div style="font-size:15px; color:#666;Observar cómo cambia el hidrógeno line-height:1.5;">
    Explorar los datos recogidos por LIGO para encontrar la diferencia entre eventos.
  </div>

</div>

## 1. ¿Cómo pueden dos detectores reconocer el mismo evento?

Antes de tocar cualquier dato, discutan en equipo (dos o tres frases por pregunta son suficientes):

- Si una misma onda gravitacional pasa por dos detectores separados por miles de kilómetros, ¿debería aparecer un rasgo relacionado en ambos?
- ¿Tiene que llegar exactamente al mismo tiempo a los dos sitios? ¿Por qué sí o por qué no?
- ¿Por qué podrían diferir la amplitud o el signo medidos en cada detector?
- ¿Qué haría que un rasgo visto en **dos** detectores fuera más convincente que uno visto en uno solo?

**Su respuesta (escriban aquí, antes de continuar):**

*(No busquen el valor publicado del retraso todavía — lo compararán con su propia medición hasta la Sesión 3.)*

## 2. Conozcamos los datos de los detectores

In [2]:
# Codigo provisto: importar paquetes y cargar el archivo preparado.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from gw_helpers import make_spectrogram, correlation_for_shift, shift_for_plot

data = np.load("student_data/gw150914_prepared.npz")

In [6]:
datapd = pd.DataFrame({key: data[key] for key in data.files})

datapd.head()

,time_from_event_s,h1_raw_strain,l1_raw_strain,h1_conditioned,l1_conditioned,sample_rate_hz,event_gps
0,-16.400000,2.177040e-19,-1.042900e-18,-9.755751,14.825122,4096.0,1.126259e+09
1,-16.399756,2.087639e-19,-1.035863e-18,-348.650083,245.890022,4096.0,1.126259e+09
2,-16.399512,2.396812e-19,-9.893224e-19,-647.082406,446.449065,4096.0,1.126259e+09
3,-16.399268,2.286722e-19,-9.375604e-19,-873.962176,593.017092,4096.0,1.126259e+09
4,-16.399024,2.132240e-19,-9.417111e-19,-1013.842812,673.945423,4096.0,1.126259e+09


In [7]:
print("Arrays disponibles:", list(data.keys()))

t = data["time_from_event_s"]
h1_raw = data["h1_raw_strain"]
l1_raw = data["l1_raw_strain"]
h1_conditioned = data["h1_conditioned"]
l1_conditioned = data["l1_conditioned"]
sample_rate_hz = float(data["sample_rate_hz"])

print(f"Frecuencia de muestreo: {sample_rate_hz} Hz")
print(f"Numero de muestras: {t.size}")
print(f"Duracion del registro: {t[-1] - t[0]:.2f} s")

Arrays disponibles: ['time_from_event_s', 'h1_raw_strain', 'l1_raw_strain', 'h1_conditioned', 'l1_conditioned', 'sample_rate_hz', 'event_gps']
Frecuencia de muestreo: 4096.0 Hz
Numero de muestras: 131072
Duracion del registro: 32.00 s


In [ ]:
# TODO: calculen el tiempo entre muestras consecutivas (dt) a partir de
# sample_rate_hz, y expresenlo tambien en milisegundos.
#
# dt = 1 / f_s

dt = ...  # <- reemplacen "..." por la expresion correcta
dt_ms = ...  # <- expresen dt en milisegundos

print(f"dt = {dt_ms:.3f} ms")

**Piensen:** a esta frecuencia de muestreo, ¿por qué se puede distinguir un retraso de varios milisegundos entre detectores? ¿Cuántas muestras corresponden aproximadamente a 7 ms?

## 3. De un strain ruidoso a un chirrido visible

In [ ]:
# Codigo provisto: intervalo fijo de graficado alrededor del evento.
t_min_plot = -0.20
t_max_plot = 0.05

mask_plot = (t >= t_min_plot) & (t <= t_max_plot)

In [ ]:
# Codigo provisto: plantilla de graficado de 4 paneles.

def plot_raw_vs_conditioned(t, h1_raw, l1_raw, h1_cond, l1_cond, mask):
    fig, axes = plt.subplots(2, 2, figsize=(11, 6), sharex=True)

    axes[0, 0].plot(t[mask], h1_raw[mask], color="C0", linewidth=0.7)
    axes[0, 0].set_title("H1 crudo")

    axes[0, 1].plot(t[mask], l1_raw[mask], color="C1", linewidth=0.7)
    axes[0, 1].set_title("L1 crudo")

    axes[1, 0].plot(t[mask], h1_cond[mask], color="C0")
    axes[1, 0].set_title("H1 condicionado")

    axes[1, 1].plot(t[mask], l1_cond[mask], color="C1")
    axes[1, 1].set_title("L1 condicionado")

    for ax in axes[1, :]:
        ax.set_xlabel("Tiempo desde el evento (s)")
    for ax in axes[:, 0]:
        ax.set_ylabel("Strain")

    fig.tight_layout()
    return fig

plot_raw_vs_conditioned(t, h1_raw, l1_raw, h1_conditioned, l1_conditioned, mask_plot)
plt.show()

**Piensen y respondan:**

- ¿Es obvia la señal en los paneles crudos?
- ¿Qué oscilación se vuelve visible después de condicionar?
- ¿Los picos de esa oscilación se acercan entre sí con el tiempo?
- ¿Por qué los paneles condicionados no deben describirse como "strain crudo"?

## 4. El chirrido en tiempo y frecuencia

In [ ]:
# Codigo provisto: espectrograma con parametros fijos (make_spectrogram),
# usando la misma escala de tiempo/frecuencia/color para H1 y L1.

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

for ax, label, x in zip(axes, ["H1", "L1"], [h1_conditioned, l1_conditioned]):
    x_windowed = x[mask_plot]
    freqs, times_spec, Sxx = make_spectrogram(x_windowed, sample_rate_hz)
    times_spec_aligned = times_spec + t[mask_plot][0]

    ax.pcolormesh(times_spec_aligned, freqs, Sxx, shading="auto")
    ax.set_title(f"Espectrograma {label}")
    ax.set_xlabel("Tiempo desde el evento (s)")

axes[0].set_ylabel("Frecuencia (Hz)")
fig.tight_layout()
plt.show()

**Piensen y respondan:**

- ¿Qué rasgo se mueve hacia arriba en frecuencia con el tiempo?
- ¿Aparece un patrón similar, de forma independiente, en H1 y en L1?

**Esta es su primera figura lista para presentación:** el chirrido condicionado y su aumento de frecuencia en ambos detectores. Guarden una interpretación corta.

**Fin de la Sesión 1.**

## 5. Midiendo el retraso entre detectores

### Convenio de signos

Los dos detectores tienen orientaciones distintas respecto a la onda que llega. Para este evento, eso significa que sus respuestas medidas tienen **signo opuesto** — no es un error de ningún instrumento, sino la orientación relativa de los interferómetros.

In [ ]:
# Codigo provisto: convenio de signos fijo para este evento.
l1_for_comparison = -l1_conditioned

**Se invirtió la respuesta de Livingston (L1).** Anoten explícitamente en su reporte que fue L1 la serie invertida.

In [ ]:
# TODO: construyan la mascara booleana para la ventana de evento fija,
# usando los limites dados, y extraigan h1_event y l1_event_for_comparison.
#
# Ventana de evento (ya decidida y probada por el equipo del proyecto):
t_min_event = -0.12
t_max_event = 0.03

mask_event = ...  # <- construyan la mascara booleana con t, t_min_event, t_max_event

t_event = t[mask_event]
h1_event = ...              # <- h1_conditioned recortado con mask_event
l1_event_for_comparison = ...  # <- l1_for_comparison recortado con mask_event

In [ ]:
# Rango de desplazamientos de prueba, aproximadamente -10 ms a +10 ms.
trial_shifts = np.arange(-41, 42)

# TODO: para cada desplazamiento de prueba, calculen la correlacion entre
# h1_event y l1_event_for_comparison usando correlation_for_shift, y
# guarden los resultados en el arreglo "correlations".
#
# Sugerencia: usen un loop (for shift in trial_shifts: ...) o una list
# comprehension, y llamen a correlation_for_shift(h1_event, l1_event_for_comparison, shift)
# en cada iteracion.

correlations = ...  # <- reemplacen por su calculo; debe quedar como np.ndarray
                     #    con la misma longitud que trial_shifts

In [ ]:
# TODO: encuentren el indice del desplazamiento con la correlacion mas alta
# (np.argmax), el desplazamiento en muestras correspondiente, y conviertan
# ese desplazamiento a milisegundos.
#
# Delta_t_ms = 1000 * N_shift / f_s

best_index = ...            # <- np.argmax(correlations)
best_shift_samples = ...    # <- trial_shifts[best_index]
best_shift_ms = ...          # <- conversion a milisegundos

print(f"Mejor desplazamiento: {best_shift_samples} muestras = {best_shift_ms:.3f} ms")

In [ ]:
# Codigo provisto: figura de correlacion contra retraso de prueba.

def plot_correlation_curve(trial_shifts, sample_rate_hz, event_correlations,
                            control_correlations=None, best_shift_samples=None):
    delays_ms = 1000 * trial_shifts / sample_rate_hz

    fig, ax = plt.subplots(figsize=(7, 4))
    ax.plot(delays_ms, event_correlations, label="Evento", color="C0")

    if control_correlations is not None:
        ax.plot(delays_ms, control_correlations, label="Control (ruido)", color="C1")

    ax.axvline(0, color="grey", linestyle="--", linewidth=1, label="retraso cero")

    if best_shift_samples is not None:
        best_delay_ms = 1000 * best_shift_samples / sample_rate_hz
        ax.axvline(best_delay_ms, color="C0", linestyle=":",
                    label=f"mejor retraso evento ({best_delay_ms:.2f} ms)")

    ax.set_xlabel("Retraso de prueba (ms)  -  positivo = Livingston primero")
    ax.set_ylabel("Correlacion de Pearson")
    ax.legend()
    fig.tight_layout()
    return fig

plot_correlation_curve(trial_shifts, sample_rate_hz, correlations,
                        best_shift_samples=best_shift_samples)
plt.show()

**Piensen y respondan:**

- ¿Hay un retraso preferido claro?
- ¿Es positivo o negativo? Según el convenio ($\Delta t_{HL} = t_{H1} - t_{L1}$, positivo = Livingston primero), ¿qué detector recibió la señal primero?
- ¿Por qué no deberían reportar más precisión que la que permite una muestra completa (~0.244 ms)?

In [ ]:
# Codigo provisto: alineen las formas de onda usando el retraso medido
# (shift_for_plot no envuelve el arreglo circularmente).

l1_aligned = shift_for_plot(l1_for_comparison, best_shift_samples)

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(t, h1_conditioned, label="H1 condicionado", alpha=0.8)
ax.plot(t, l1_aligned, label="L1 invertido y alineado", alpha=0.8)
ax.set_xlim(t_min_event, t_max_event)
ax.set_xlabel("Tiempo desde el evento (s)")
ax.set_ylabel("Strain condicionado")
ax.legend()
fig.tight_layout()
plt.show()

**Piensen:** ¿las oscilaciones principales se alinean mejor después de aplicar el desplazamiento medido?

**Fin de la Sesión 2:** deberían tener el retraso medido (en muestras y en milisegundos), la figura de correlación y esta superposición alineada.

## 6. ¿El ruido ordinario da el mismo resultado?

In [ ]:
# TODO: repitan exactamente el mismo procedimiento de la seccion 5, pero
# en este intervalo de control fijo, que no deberia contener el evento.
#
# Intervalo de control (misma duracion que el intervalo de evento):
t_min_control = -5.00
t_max_control = -4.85

mask_control = ...  # <- mascara booleana con t, t_min_control, t_max_control

t_control = t[mask_control]
h1_control = ...                    # <- h1_conditioned recortado con mask_control
l1_control_for_comparison = ...     # <- l1_for_comparison recortado con mask_control

# Repitan el loop de correlacion para trial_shifts sobre el intervalo de control:
control_correlations = ...  # <- mismo procedimiento que "correlations" en la seccion 5

print(f"Correlacion maxima (evento):  {correlations.max():.3f}")
print(f"Correlacion maxima (control): {control_correlations.max():.3f}")

In [ ]:
plot_correlation_curve(trial_shifts, sample_rate_hz, correlations,
                        control_correlations=control_correlations,
                        best_shift_samples=best_shift_samples)
plt.show()

**Esta es su segunda figura lista para presentación.**

**Piensen y respondan:**

- Todo intervalo escaneado tiene *algún* máximo. ¿Es el resultado del evento claramente más convincente que esta comparación de ruido ordinario?
- Esto **no** es un cálculo formal de significancia estadística — es una comparación de tipo control de clase. ¿Por qué es útil de todas formas?

## 7. ¿Es el retraso físicamente posible?

In [ ]:
# TODO: calculen el tiempo maximo de viaje de la luz entre los dos sitios
# (separados aproximadamente D_km = 3002 km), usando t_max = D / c.

D_km = 3002
c_km_s = 299_792

t_max_ms = ...  # <- calculen 1000 * D_km / c_km_s

print(f"t_max = {t_max_ms:.2f} ms")
print(f"Retraso medido = {best_shift_ms:.2f} ms")

**Piensen y respondan:**

- ¿El retraso medido es físicamente posible para una única señal viajando a la velocidad de la luz?
- ¿Un retraso menor a 10 ms *prueba* que la señal es astrofísica? ¿Es necesario, suficiente, o ambos?
- ¿Por qué un solo retraso medido entre dos detectores no determina una única dirección en el cielo?

### Comparen con el resultado publicado

Solo ahora, después de tener su propia medición, comparen con el valor publicado por LIGO:

$$\Delta t_{\mathrm{HL}} = 6.9^{+0.5}_{-0.4}\ \mathrm{ms}$$

Como la medición está limitada a muestras completas, las dos respuestas más cercanas posibles son 28 muestras (≈6.84 ms) y 29 muestras (≈7.08 ms). Que su resultado esté **cerca** de este rango importa mucho más que reproducir "6.9" de forma exacta.

## 8. ¿Qué podemos concluir?

### Construyan su cadena de evidencia

Usando sus propias figuras y números, completen (en sus palabras) los pasos que correspondan:

1. El evento (¿es o no es?) obvio en el strain crudo.
2. El condicionamiento revela una oscilación cuya frecuencia ______ con el tiempo.
3. Un chirrido parecido aparece en ______ detectores.
4. Las dos respuestas de detector se alinean mejor cuando una se ______ y se ______.
5. La correlación identifica un retraso preferido de aproximadamente ______ milisegundos.
6. Ese retraso indica que ______ recibió la señal primero.
7. El retraso está ______ (por encima / por debajo) del tiempo máximo de viaje de la luz entre los sitios.
8. La comparación del evento es ______ (más / menos) convincente que el control de solo ruido.

### Tres niveles de afirmación

Clasifiquen las siguientes frases según el nivel que les corresponde y agreguen al menos un ejemplo propio en cada nivel:

- **Apoyado directamente por su trabajo.**
- **Apoyado por el análisis público completo de LIGO** (no por este proyecto).
- **No establecido por este proyecto.**

### Al menos una limitación

Escriban al menos una limitación explícita de su análisis (por ejemplo: un único intervalo de control, sin significancia estadística formal, sin descartar toda posible causa instrumental, etc.).

### Checklist final

- [ ] Figura del chirrido (Sección 4)
- [ ] Figura de correlación evento + control, con el mejor retraso marcado (Sección 6)
- [ ] Retraso estimado en muestras y en milisegundos
- [ ] Orden de llegada
- [ ] Tiempo máximo de viaje de la luz
- [ ] Conclusión ligada a las figuras y números
- [ ] Al menos una limitación explícita